# Extra Theory Experiments | correction_completion | shards 070+071/384

Use at most two workers on a 12 GB Colab session; default is one. This single-wave notebook runs two original shards sequentially (070 then 071), each into its own original shard output directory, so aggregation with --n-shards 384 is unchanged and each notebook downloads its shard archives plus one combined summary. Run the notebook once; no ETE_N_WAVES or ETE_WAVE_ID configuration is needed. Replication counts follow each design's config value (ETE_REPS overrides).

In [ ]:
# Colab bootstrap: branch is pinned so this notebook is self-contained.
import os, sys, pathlib, subprocess
REPO_URL = 'https://github.com/hugogobato/DiD-BCF.git'
BRANCH = 'main'
TARGET = pathlib.Path("DiD-BCF")
if not (TARGET / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(TARGET)], check=True)
else:
    # Refresh an existing disposable clone so reruns cannot keep stale code.
    subprocess.run(["git", "-C", str(TARGET), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(TARGET), "reset", "--hard", f"origin/main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(TARGET / "Simulation_Studies_Revision" / "Theory_Calibration" / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(TARGET))
sys.path.insert(0, str(TARGET / "Simulation_Studies_Revision" / "Theory_Calibration" / "src"))
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
print("Using clone:", TARGET.resolve(), "| max workers: 2, default workers: 1")


In [ ]:
from extra_theory_experiments.manifest import build_manifest, manifest_frame
from extra_theory_experiments.runner import run_tasks, write_provenance
import json, os, zipfile

import pandas as pd

FAMILY = 'correction_completion'
SHARD_GROUP = (70, 71)
SHARD_IDS = list(SHARD_GROUP)
N_SHARDS = 384
N_WAVES = 1
WAVE_ID = 0
REPS = int(os.environ['ETE_REPS']) if os.environ.get('ETE_REPS', '').strip() not in ('', '0') else None
SMOKE = os.environ.get("ETE_SMOKE", "0") == "1"
BCF_PARAMS = {'num_gfr': 50, 'num_mcmc': 500, 'keep_every': 5, 'num_chains': 3}

RESULTS_ROOT = TARGET / "Simulation_Studies_Revision" / "Theory_Calibration" / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVES = []
SUMMARIES = []
for SHARD_ID in SHARD_IDS:
    print(f"[correction_completion] running original shard {SHARD_ID:03d}/{N_SHARDS:03d}", flush=True)
    OUT = str(RESULTS_ROOT / f"extra_theory_correction_completion_shard_{SHARD_ID:03d}")
    os.makedirs(OUT, exist_ok=True)
    tasks = build_manifest(FAMILY, reps=REPS, n_shards=N_SHARDS, shard_id=SHARD_ID,
                           wave_id=WAVE_ID, n_waves=N_WAVES)
    manifest_frame(tasks).to_csv(os.path.join(OUT, "manifest.csv"), index=False)
    summary = run_tasks(tasks, out_dir=os.path.join(OUT, "checkpoints"),
                        bcf_params=BCF_PARAMS, smoke=SMOKE, resume=True)
    summary.to_csv(os.path.join(OUT, "summary.csv"), index=False)
    try:
        summary.to_parquet(os.path.join(OUT, "summary.parquet"), index=False)
    except Exception as exc:
        print("Parquet skipped:", exc)
    write_provenance(os.path.join(OUT, "provenance.json"), tasks=tasks,
                     repo_root=str(TARGET), smoke=SMOKE,
                     bcf_params=BCF_PARAMS,
                     config={"family": FAMILY, "reps": REPS, "shard": SHARD_ID,
                             "n_shards": N_SHARDS, "wave_id": WAVE_ID,
                             "n_waves": N_WAVES, "K": 2, "workers": 1,
                             "pilot": None})
    with open(os.path.join(OUT, "README_run.txt"), "w", encoding="utf-8") as handle:
        handle.write("Family=" + FAMILY + "; shard=" + str(SHARD_ID) + "/" + str(N_SHARDS) +
                     "; wave=" + str(WAVE_ID) + "/" + str(N_WAVES) + "; reps=" + str(REPS) +
                     "; smoke=" + str(SMOKE) + "\n")
    archive_path = OUT + ".zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for root, _, names in os.walk(OUT):
            for name in names:
                path = os.path.join(root, name)
                archive.write(path, arcname=os.path.relpath(path, OUT))
    print("Wrote archive:", archive_path)
    ARCHIVES.append(archive_path)
    SUMMARIES.append(summary)
combined = pd.concat(SUMMARIES, ignore_index=True) if SUMMARIES else pd.DataFrame()
combined_path = str(RESULTS_ROOT /
                    f"extra_theory_correction_completion_shards_{SHARD_IDS[0]:03d}_{SHARD_IDS[-1]:03d}_summary.csv")
combined.to_csv(combined_path, index=False)
print("Wrote combined summary:", combined_path)
OUTPUT_FILES = ARCHIVES + [combined_path]


In [ ]:
try:
    from google.colab import files
    for output_file in OUTPUT_FILES:
        files.download(output_file)
        print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
